In [27]:
import sys
from pathlib import Path
from dotenv import load_dotenv
# Production layout: add project root and src for imports (run from repo root or notebooks/ingestion/)
_root = Path(".").resolve()
if _root.name == "ingestion":
    _root = _root.parent.parent
elif (_root / "src").is_dir():
    pass
else:
    _root = _root.parent
load_dotenv(_root / ".env")
load_dotenv("/app/.env")
sys.path.insert(0, str(_root))
sys.path.insert(0, str(_root / "src"))

from storage.postgres.pgConn import PgConn
from storage.postgres import PostgresSQL_table_queries
from storage.postgres.news_dataframe import (
    filter_financial_news_by_date,
    filter_financial_news_ingested_today,
    filter_financial_news_published_today,
    get_financial_news_content_by_id,
    normalize_financial_news_datetime_column,
)
from storage.cloud.CloudStorage import CloudStorageProvider

import pandas as pd
from datetime import date, datetime

# Sanity check: if this fails, use File → Reload Notebook from Disk, then restart kernel
import storage.postgres.news_dataframe as _news_df
print(f"Using news_dataframe from: {_news_df.__file__}")
print(f"filter_financial_news_ingested_today: OK")

Using news_dataframe from: /app/src/storage/postgres/news_dataframe.py
filter_financial_news_ingested_today: OK


In [28]:
table_name = PostgresSQL_table_queries.FINANCIAL_NEWS_TABLE_NAME
pg_conn = PgConn(table_name)
df = pg_conn.get_financial_news()
if df is None:
    raise RuntimeError(
        "get_financial_news() failed — check the error printed above "
        "(connection, table name, or schema)."
    )

Connection to the database successful!
Table name set to: financial_news_241118


In [29]:
print(f"Total news articles queried: {df.shape[0]}")

Total news articles queried: 99


In [30]:
df.head()

,id,source,headline,href,summary,content,author,minsread,datetime,created_at
0,1252715375008582536,24/7 Wall St.,"$10,000 in XRP vs $10,000 in HYPE: Which Is Wo...",https://finance.yahoo.com/markets/crypto/artic...,,"Quick Read\nHYPE has turned a $10,000 entry si...",Sam Daodu,8 min read,2026-08-07 21:30:14,2026-08-08 13:50:14.923590
1,3319435024810627900,Motley Fool,1 Unstoppable Cryptocurrency to Buy Before It ...,https://finance.yahoo.com/markets/crypto/artic...,,"Bitcoin (CRYPTO: BTC) may be down, but it's no...","Dominic Basulto, The Motley Fool",3 min read,2026-08-07 10:06:00,2026-08-08 13:50:15.083948
2,1133790260607610103,BeInCrypto,"161,000 Idle ETH May Soon Be Staked: What Gray...",https://finance.yahoo.com/markets/crypto/artic...,,Hands holding an Ethereum coin as the Grayscal...,Lockridge Okoth,2 min read,2026-08-07 15:43:14,2026-08-08 13:50:14.989297
3,2549688907357340228,Motley Fool,3 Cryptocurrencies Making Bold Moves in AI Tha...,https://finance.yahoo.com/markets/crypto/artic...,,The age of artificial intelligence (AI) is her...,"Alex Carchidi, The Motley Fool",5 min read,2026-08-07 10:10:00,2026-08-08 13:50:15.080353
4,2167476789037798255,Cryptonews,A Coldcard Hacker Just Moved $1.94 Million in ...,https://finance.yahoo.com/markets/crypto/artic...,,"In the latest Bitcoin news, a wallet associate...",Ahmed Barakat,3 min read,2026-08-07 15:28:00,2026-08-08 13:50:14.992036


In [31]:
def delete_records_for_current_date(df, pg_conn):
    if df is None:
        print("DataFrame is empty. No records to delete.")
        return
        
    today_records = filter_financial_news_by_date(df)

    # Extract date strings (DB/delete API expects stored string form)
    date_strings = today_records['datetime'].astype(str).tolist()

    # Call the delete_records_by_date method
    pg_conn.delete_records_by_date(date_strings)

# Then call the delete_records_for_current_date method
#delete_records_for_current_date(df, pg_conn)
#list_ids = ["11111"]
#pg_conn.delete_records_by_ids(list_ids)

In [32]:
class DataETL():
    
    def __init__(self, dataframe):
            self.df = dataframe
            self.cloudProvider = CloudStorageProvider()
            
    class Export():
        def __init__(self, dataframe):
            self.df = dataframe
            self.cloudProvider = CloudStorageProvider()
            
        def set_dataframe(self, dataframe):
            self.df = dataframe
        
        def export_text_to_s3(self, bucket_name, prefix_path, file_format):
            # Initialize AWS storage
            aws_storage = self.cloudProvider.AWS()

            # Create a new bucket
            aws_storage.create_bucket(bucket_name)

            # Upload DataFrame with datetime subfolder structure
            aws_storage.upload_dataframe_with_datetime_subfolders(self.df, bucket_name, prefix_path, file_format)
        
        def export_text_to_s3_full_file(self, bucket_name, prefix_path, filename):
            aws_storage = self.cloudProvider.AWS()
            aws_storage.upload_dataframe_to_csv(self.df, bucket_name, filename, prefix_path)
            
    class Ingestion():
        def __init__(self, dataframe):
            self.df = dataframe
            self.cloudProvider = CloudStorageProvider()
            
        def get_full_data_csv_file(self, bucket_name, prefix_path):
            # Initialize AWS storage
            aws_storage = self.cloudProvider.AWS()
            return aws_storage.get_csv_from_specific_folder(bucket_name, prefix_path)
        
        def get_data_csv_file_by_datetime(
            self, bucket_name, prefix_path, year, month, day, hour=None, minute=None, second=None
        ):
            aws_storage = self.cloudProvider.AWS()
            return aws_storage.get_dataframe_from_specific_datetime(
                bucket_name,
                prefix_path,
                year=year,
                month=month,
                day=day,
                hour=hour,
                minute=minute,
                second=second,
            )
    
    class Process():
        
        def __init__(self, dataframe):
            self.df = dataframe
        
        def getData():
            self.df = pg_conn.get_financial_news()
        
        def filter_by_current_date(self):
            # Article publish date (datetime) on today's calendar date (default on_date=None)
            return filter_financial_news_by_date(self.df)
            
    class Transform():
        def extractStopWords():
            pass

In [33]:
etl = DataETL(df)

# Export by article publish time (datetime). on_date=None → today's local date (date.today()).
export_on_date = "2026-08-08"  # e.g. "2026-05-23" to override today
filtered_df = filter_financial_news_by_date(df, on_date=export_on_date)

print(
    f"Filter date (datetime column): {export_on_date or date.today()} | "
    f"Rows matched: {len(filtered_df)} | "
    f"Ingested today (created_at only): {len(filter_financial_news_ingested_today(df))}"
)
filtered_df.head()

Filter date (datetime column): 2026-08-08 | Rows matched: 8 | Ingested today (created_at only): 99


,id,source,headline,href,summary,content,author,minsread,datetime,created_at
18,1163886604058749035,CCN,"Bitcoin, Ethereum and XRP Face a CLARITY Act S...",https://finance.yahoo.com/markets/crypto/artic...,,Key Takeaways\nFour leading AI models expect t...,Giuseppe Ciccomascolo,4 min read,2026-08-08 09:17:38,2026-08-08 13:50:14.883050
20,830138900716863910,TheStreet,Bybit takes North Korea and Lazarus Group to c...,https://finance.yahoo.com/markets/crypto/artic...,,Cryptocurrency exchange Bybit has taken North ...,Neo,2 min read,2026-08-08 01:17:00,2026-08-08 13:50:14.892440
27,358459614784406425,Simply Wall St.,Circle Internet Group (CRCL) Launches Arc As I...,https://finance.yahoo.com/markets/crypto/artic...,,Get insights on thousands of stocks from the g...,Bailey Pemberton,2 min read,2026-08-08 04:19:23,2026-08-08 13:50:14.889788
33,2590782451210276043,Motley Fool,Crypto Exchanges Are Seeing Less Activity. Her...,https://finance.yahoo.com/markets/crypto/artic...,,Trading on the world's biggest crypto exchange...,"Alex Carchidi, The Motley Fool",5 min read,2026-08-08 09:50:00,2026-08-08 13:50:14.873986
46,793399664898327956,BeInCrypto,Jim Cramer Names 5 Investing Themes and 13 Sto...,https://finance.yahoo.com/markets/stocks/artic...,,Photo by BeInCrypto\nJim Cramer named 5 invest...,Kamina Bashir,2 min read,2026-08-08 09:19:53,2026-08-08 13:50:14.876756


In [34]:
print(f"Total filtered news articles queried: {filtered_df.shape[0]}")

Total filtered news articles queried: 8


In [35]:
import os

export_rows_to_s3 = True
etl_export = etl.Export(filtered_df)
bucket_name = "test-financial-news-bucket"
prefix_path = "news/crypto"
file_format = "csv"

if export_rows_to_s3 and not filtered_df.empty:
    if not os.getenv("AWS_ACCESS_KEY_ID") or not os.getenv("AWS_SECRET_ACCESS_KEY"):
        raise RuntimeError(
            "AWS credentials not configured. Uncomment and set AWS_ACCESS_KEY_ID and "
            "AWS_SECRET_ACCESS_KEY in .env, then restart Jupyter: ./docker/start_jupyter.ps1"
        )
    export_df = normalize_financial_news_datetime_column(filtered_df)
    etl_export.set_dataframe(export_df)
    etl_export.export_text_to_s3(bucket_name, prefix_path, file_format)

Bucket 'test-financial-news-bucket' already exists.
Data for row 18 with id '1163886604058749035' uploaded to S3 bucket 'test-financial-news-bucket' under folder 'news/crypto/year=2026/month=08/day=08/hour=09/minute=17/second=38/format=csv/1163886604058749035.csv'
Data for row 20 with id '830138900716863910' uploaded to S3 bucket 'test-financial-news-bucket' under folder 'news/crypto/year=2026/month=08/day=08/hour=01/minute=17/second=00/format=csv/830138900716863910.csv'
Data for row 27 with id '358459614784406425' uploaded to S3 bucket 'test-financial-news-bucket' under folder 'news/crypto/year=2026/month=08/day=08/hour=04/minute=19/second=23/format=csv/358459614784406425.csv'
Data for row 33 with id '2590782451210276043' uploaded to S3 bucket 'test-financial-news-bucket' under folder 'news/crypto/year=2026/month=08/day=08/hour=09/minute=50/second=00/format=csv/2590782451210276043.csv'
Data for row 46 with id '793399664898327956' uploaded to S3 bucket 'test-financial-news-bucket' unde

In [36]:
post_full_csv = False
if (post_full_csv == True) and not filtered_df.empty:
    now = datetime.now()
    filename = f"{now.year}-{now.month:02}-{now.day:02}_full_record"
    etl_export.set_dataframe(etl.df)
    etl_export.export_text_to_s3_full_file(bucket_name, prefix_path, filename)

In [37]:
ingest_data = False
get_full_file = False
get_by_datetime = True
df_from_file = None

if ingest_data == True:
    etl_ingestion = etl.Ingestion(etl.df)
    bucket_name = "test-financial-news-bucket"
    prefix_path = "news/crypto/"
    year = '2026'
    month = '05'
    day = '24'
    hour = None   # set e.g. '04' to narrow to one hour; None = whole day
    minute = None
    if get_full_file == True:
        filename = f"{year}-{month}-{day}_full_record.csv"
        full_path = f"{prefix_path}{filename}"
        df_from_file = etl_ingestion.get_full_data_csv_file(bucket_name, full_path)
    elif get_by_datetime == True:
        df_from_file = etl_ingestion.get_data_csv_file_by_datetime(bucket_name, prefix_path, year, month, day, hour, minute)

In [38]:
if df_from_file is not None:
    print(df_from_file.count())
    df_from_file.head()

In [39]:
targetId = ""  # i.e. "1221589746717124508" targetId can be string or numeric type

# Lookup order: S3 ingest result, filtered export batch, then full DB pull
lookup_df = None
lookup_source = None
for name, candidate in (
    ("df_from_file", df_from_file if "df_from_file" in dir() else None),
    ("filtered_df", filtered_df if "filtered_df" in dir() else None),
    ("df", df if "df" in dir() else None),
):
    if candidate is not None and not getattr(candidate, "empty", True):
        lookup_df = candidate
        lookup_source = name
        break

if targetId and lookup_df is not None:
    full_content = get_financial_news_content_by_id(lookup_df, targetId)
    if full_content:
        print(full_content)
    else:
        print(
            f"No content for id {targetId!r} in {lookup_source} "
            f"({len(lookup_df)} rows). Id column dtype: {lookup_df['id'].dtype}"
        )
else:
    print("Set targetId and ensure df_from_file, filtered_df, or df is loaded.")

Set targetId and ensure df_from_file, filtered_df, or df is loaded.
